# 03. ZeRO Stages 1, 2, 3 and FSDP2 Fully Shard

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Rajbhandari et al. (DeepSpeed ZeRO, SC 2020, *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models*), Zhao et al. (PyTorch FSDP, 2023), PyTorch Distributed Team (FSDP2, 2024).
- **Purpose:** Eliminate static state memory redundancies by progressively sharding optimizer states (ZeRO-1), gradients (ZeRO-2), and model parameters (ZeRO-3 / FSDP2) across data-parallel ranks.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Device Mesh and DTensor Sharding Placements](02_device_mesh_and_dtensor_sharding_placements.ipynb) $\to$ `🔗 Downstream Syntheses:` [Tensor Parallelism Megatron Sequence and Async TP](04_tensor_parallelism_megatron_sequence_and_async_tp.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use ZeRO-3 / FSDP2 when per-GPU batch sizes and network bandwidth cannot overlap the $+50\%$ extra AllGather communication ($3S\beta$ vs $2S\beta$). If parameters fit comfortably in HBM, prefer ZeRO-1 or ZeRO-2.

## Mathematical Foundations & Governing Equations

$$
M_{\text{DDP}} = 16\Psi, \quad M_{\text{ZeRO-1}} = 4\Psi + \frac{12\Psi}{N_d}, \quad M_{\text{FSDP2}} = \frac{16\Psi}{N_d}, \quad T_{\text{FSDP2}} = 3 S \beta
$$

### Asymptotic Time & Memory Complexity
- **Memory Reduction:** Up to $16\times$ static state reduction across $N_d$ data-parallel ranks.
- **Communication Overhead:** Exactly $1.5\times$ baseline DDP volume, completely hidden via asynchronous forward/backward AllGather prefetching.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def fsdp2_forward_matmul(x: torch.Tensor, param_shards: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    world_size = param_shards.shape[0]
    w_full = torch.cat([param_shards[r] for r in range(world_size)], dim=0)
    return x @ w_full, w_full

def fsdp2_backward_reducescatter(x: torch.Tensor, grad_out: torch.Tensor, world_size: int) -> tuple[torch.Tensor, torch.Tensor]:
    grad_w_full = x.t() @ grad_out
    chunks = torch.chunk(grad_w_full, world_size, dim=0)
    return grad_w_full, torch.stack(chunks, dim=0)

def zero1_adam_step(p: torch.Tensor, g: torch.Tensor, m: torch.Tensor, v: torch.Tensor, lr: float = 0.01, step: int = 1) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    m_new = 0.9 * m + 0.1 * g
    v_new = 0.999 * v + 0.001 * (g ** 2)
    m_hat = m_new / (1.0 - 0.9 ** step)
    v_hat = v_new / (1.0 - 0.999 ** step)
    p_new = p - lr * m_hat / (torch.sqrt(v_hat) + 1e-8)
    return p_new, m_new, v_new

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

h, w_full = fsdp2_forward_matmul(weights["x"], weights["param_shards"])
assert float(torch.max(torch.abs(w_full - weights["w_full"]))) < 1e-5
assert float(torch.max(torch.abs(h - weights["fsdp_fwd_out"]))) < 1e-5

grad_w_full, grad_w_shards = fsdp2_backward_reducescatter(weights["x"], weights["grad_out"], 2)
assert float(torch.max(torch.abs(grad_w_full - weights["grad_w_full"]))) < 1e-5
assert float(torch.max(torch.abs(grad_w_shards - weights["grad_w_shards"]))) < 1e-5

p_new, m_new, v_new = zero1_adam_step(weights["p_init"], weights["g_init"], weights["m_init"], weights["v_init"], lr=0.01, step=1)
assert float(torch.max(torch.abs(p_new - weights["p_new"]))) < 1e-5
assert float(torch.max(torch.abs(m_new - weights["m_new"]))) < 1e-5
assert float(torch.max(torch.abs(v_new - weights["v_new"]))) < 1e-5
print("All FSDP2 Forward, Backward, and ZeRO-1 Adam checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: fsdp2_forward_matmul(weights["x"], weights["param_shards"]), iters=50, name="FSDP2 Forward Matmul"),
    benchmark_fn(lambda: fsdp2_backward_reducescatter(weights["x"], weights["grad_out"], 2), iters=50, name="FSDP2 Backward ReduceScatter"),
    benchmark_fn(lambda: zero1_adam_step(weights["p_init"], weights["g_init"], weights["m_init"], weights["v_init"]), iters=50, name="ZeRO-1 Adam Step"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Replicated Optimizer Redundancy vs ZeRO-1 Partitioning
num_params = 1_000_000_000  # 1 Billion parameters
world_size = 8

# Baseline DDP memory consumption per GPU (FP16 params + grads + FP32 Adam states):
# 2 bytes (params) + 2 bytes (grads) + 4 bytes (master p) + 4 bytes (m) + 4 bytes (v) = 16 bytes/param
ddp_mem_gb = (16 * num_params) / (1024 ** 3)

# ZeRO-1 partitions the 12 bytes/param of Adam states across world_size ranks:
zero1_mem_gb = (4 * num_params + (12 * num_params) / world_size) / (1024 ** 3)

# FSDP2 / ZeRO-3 partitions all 16 bytes/param across world_size ranks:
fsdp2_mem_gb = ((16 * num_params) / world_size) / (1024 ** 3)

print(f"Baseline DDP Static Memory:    {ddp_mem_gb:.2f} GB / GPU")
print(f"ZeRO-1 Partitioned Memory:    {zero1_mem_gb:.2f} GB / GPU  ({ddp_mem_gb / zero1_mem_gb:.1f}x reduction)")
print(f"FSDP2 / ZeRO-3 Fully Sharded: {fsdp2_mem_gb:.2f} GB / GPU  ({ddp_mem_gb / fsdp2_mem_gb:.1f}x reduction)")
assert ddp_mem_gb > zero1_mem_gb > fsdp2_mem_gb, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Zero Redundancy Supremacy:** Eliminating static parameter, gradient, and optimizer duplicates enables training models $16\times$ larger than single-GPU HBM without modifying model code.
2. **Dynamic Lifecycle Management:** FSDP2 materializes full parameters only on-demand during forward and backward passes, immediately freeing full weights to minimize peak active memory.
3. **Overlap Saturation:** Prefetching parameter AllGather operations on high-bandwidth NVLink backbones conceals the $50\%$ communication increase behind computation.

---
### Cross-Domain Generalisation Matrix
| Domain | Sharding / Partitioning Analogue | Practical Scaling Advantage |
| :--- | :--- | :--- |
| **Large-Scale Finite Element (FEM)** | Distributed subdomain matrix stiffness sharding | Distributes massive sparse stiffness matrices across supercomputer MPI ranks |
| **Distributed Key-Value Stores** | Consistent hashing range partitioning | Replaces monolithic database nodes with horizontal shard replication |
| **Particle-in-Cell (PIC) Plasma** | Spatial particle & grid decomposition | Reallocates particle buffers dynamically based on local plasma density |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_zero_stages_1_2_3_and_fsdp2_fully_shard.md](../../markdown/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.md)
- **PyTorch 2.x Module:** [03_zero_stages_1_2_3_and_fsdp2_fully_shard.py](../../pytorch2/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.py)
- **Functional JAX Module:** [03_zero_stages_1_2_3_and_fsdp2_fully_shard.py](../../jax/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.py)
- **Native Rust Candle Track:** [03_zero_stages_1_2_3_and_fsdp2_fully_shard.rs](../../candle/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.rs)
- **Hardware Kernel (CPU / WGSL):** [03_zero_stages_1_2_3_and_fsdp2_fully_shard.rs](../../kernels/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.rs)
- **Golden SafeTensors:** [03_zero_stages_1_2_3_and_fsdp2_fully_shard.safetensors](../../weights/16_distributed_and_scalable/03_zero_stages_1_2_3_and_fsdp2_fully_shard.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Device Mesh and DTensor Sharding Placements](02_device_mesh_and_dtensor_sharding_placements.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 04. Tensor Parallelism Megatron Sequence and Async TP](04_tensor_parallelism_megatron_sequence_and_async_tp.ipynb) &rarr;